# Lab 2 - Model Agnostic Methods

**How the lab works**

- The questions are divided into three levels.
  - **[Basic]**: if your group completes all Basic questions of a lab correctly, you get a 6 (a passing grade).
  - **[Regular]**: these questions give up to 2 extra points.
  - **[Advanced]**: for students looking for a challenge; expect to work on them in your own time.
- Every group works with slightly different data and settings, so your answers, numbers and plots are different from those of other groups.
  - Set your group number $G$ in the first code cell. Your seed is $S = G$; use it everywhere a random seed is needed (`random_state=S`).
  - *Your patient* is row $p = S \bmod n$ of your test set, where $n$ is the number of rows in the test set.
  - Put your group number in the title of every plot.
- Answers must use the numbers and plots from your own notebook. An answer that does not match your own output gets no points.
- You may use AI tools to help with code, but every group member must be able to explain what you hand in.
- Lines marked `# TODO` must be completed. Write your answers in the **Your answers** cells.
- Hand in this notebook with all cells run. Also convert the notebook with all cells run into PDF and hand in. 
  

**Data and models.** Use the same group number, data, split and models as in Lab 1, plus a random forest.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import PartialDependenceDisplay, permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

from xai_lab_utils import group_seed, personal_data_path, your_patient_index

In [ ]:
# Fill in your group number. Everything else depends on it.
GROUP_NUMBER = None  # TODO: your group number, e.g. 7

S = group_seed(GROUP_NUMBER)   # use S everywhere a random seed is needed
print("Group", GROUP_NUMBER, "- seed S =", S)

In [ ]:
# TODO: paste your code from Lab 1 here:
#   - reading the data file (Q1)
#   - the train/test split and the preprocessor (Q3)
#   - the three fitted models: logreg, tree, knn

In [ ]:
rf = RandomForestClassifier(n_estimators=300, random_state=S).fit(X_train, y_train)
models = {"logistic": logreg, "tree": tree, "kNN": knn, "random forest": rf}

p = your_patient_index(GROUP_NUMBER, len(X_test))
x_p = X_test.iloc[[p]]

## Q1) Permutation importance

a) **[Basic]** Before you run it, write down which three attributes you expect to have the highest permutation importance for the logistic model (look at your odds ratios from Lab 1). Then perform permutation importance for the logistic model for all attributes and plot the results. Give a brief interpretation. Was your expectation right?

b) **[Basic]** Which model is most likely to show a bigger difference in prediction when using permutation importance? Run it for all models and make a table with the top 5 attributes of each model.

c) **[Regular]** Run permutation importance for the random forest on the training set and on the test set. What is the difference? What does it tell you about the model?

d) **[Advanced]** Add a copy of `thalach` with a little noise to the data and run permutation importance again. What happens to the importance of `thalach`? Why?

e) **[Advanced]** Add a column of random numbers. Compare the built-in `feature_importances_` of the random forest with permutation importance. Which one gives importance to the random column, and why?

> **Hint:** `permutation_importance(model, X_test, y_test, n_repeats=30, random_state=S)`; `r.importances_mean`, `r.importances_std`

In [ ]:
r = permutation_importance(logreg, X_test, y_test, n_repeats=30, random_state=S)

imp = pd.DataFrame({"mean": r.importances_mean, "std": r.importances_std},
                   index=X_test.columns).sort_values("mean")
plt.figure(figsize=(7, 6))
plt.barh(imp.index, imp["mean"], xerr=imp["std"])
plt.xlabel("decrease in accuracy")
plt.title(f"Permutation importance, logistic model - group {GROUP_NUMBER}")
plt.show()

# TODO (b): run it for all models and make a table with the top 5 of each model
# TODO (c): random forest on the training set and on the test set

**Your answers:**

a) 

b) 

c) 

d) 

e) 

## Q2) Partial dependence and ICE

a) **[Basic]** Choose the two attributes you think explain the most, and motivate your choice. For each model, plot the partial dependence plot (PDP) of these attributes together with the ICE curves. Comment on the differences between the models. Do all patients behave the same way? Point to a place in your plot where the PDP hides something.

b) **[Basic]** Plot the two-way PDP of `trestbps` (resting blood pressure) and `cp` (chest pain type) and explain their effect.

c) **[Advanced]** Write your own code for the Ceteris Paribus profile (algorithm in the lecture) of your patient for `thalach` and for `cp`. Compare it with the PDP.

d) **[Advanced]** Plot the PDP of the logistic model once on the probability scale and once on the log-odds scale. Why is one of them a straight line?

> **Hint:** `PartialDependenceDisplay.from_estimator(model, X_test, features, kind="both", centered=True)`;
> two-way: `features=[("trestbps", "cp")], categorical_features=["cp"]`; log-odds scale: `response_method="decision_function"`

In [ ]:
features = ["age", "thalach"]  # TODO (a): choose the two attributes you think explain the most

for name, model in models.items():
    disp = PartialDependenceDisplay.from_estimator(
        model, X_test, features, kind="both", centered=True, random_state=S)
    disp.figure_.suptitle(f"PDP + ICE, {name} - group {GROUP_NUMBER}")
    plt.show()

In [ ]:
# TODO (b): two-way PDP of trestbps and cp
# PartialDependenceDisplay.from_estimator(model, X_test, [("trestbps", "cp")], categorical_features=["cp"])

**Your answers:**

a) 

b) 

c) 

d) 

## Q3) SHAP

*Note:* for the force plot the notebook must be trusted (`shap.initjs()`).

a) **[Basic]** Use a SHAP explainer to derive the SHAP values for the logistic model (code below). Plot a SHAP summary plot using all attributes and explain the plot. Compare the order of the attributes with permutation importance.

b) **[Basic]** Plot the SHAP dependence plot for `exang`. Explain how to read this plot, and how it differs from a PDP.

c) **[Basic]** Plot a force plot and a waterfall plot for your patient and explain the figures. Check that the base value plus the sum of the SHAP values equals the model output for your patient. Give the numbers.

d) **[Regular]** For the logistic model, the SHAP value of attribute $j$ is $\beta_j (x_j - \text{mean}(x_j))$ on the log-odds scale. Check this for your patient with your numbers.

e) **[Advanced]** Find a patient in your test set whose most important attribute (by SHAP) is not the globally most important one. Explain why this is not a contradiction.

f) **[Advanced]** Use KernelSHAP with a background of 10, 50 and 200 patients. How much do the SHAP values change? How long does each run take?

> **Hint:** `shap.plots.beeswarm(sv)`, `shap.plots.bar(sv)`, `shap.plots.scatter(sv[:, "exang"], color=sv)`;
> `shap.plots.waterfall(sv[p])`, `shap.plots.force(sv[p])`; `sv.base_values[p]`, `sv.values[p].sum()`, `clf.decision_function(Xt_test)[p]`;
> `shap.KernelExplainer(f, shap.sample(X_train, 50, random_state=S))`

In [ ]:
shap.initjs()

# SHAP for the logistic model, on the preprocessed (scaled) data.
# The values are on the log-odds scale.
pre, clf = logreg[:-1], logreg[-1]
Xt_train, Xt_test = pre.transform(X_train), pre.transform(X_test)

# Use the full training set as background (by default SHAP keeps only 100 rows)
masker = shap.maskers.Independent(Xt_train, max_samples=len(Xt_train))
explainer = shap.LinearExplainer(clf, masker)
sv = explainer(Xt_test)

shap.plots.beeswarm(sv, max_display=20)

In [ ]:
# (b) dependence plot for exang
shap.plots.scatter(sv[:, "exang"], color=sv)

In [ ]:
# (c) your patient
shap.plots.waterfall(sv[p], max_display=20)
shap.plots.force(sv[p])

# TODO (c): check that base value + sum of SHAP values = model output (log-odds)
# model output on the log-odds scale: clf.decision_function(Xt_test)[p]

**Your answers:**

a) 

b) 

c) 

d) 

e) 

f) 

## Q4) Compare two models

a) **[Basic]** Derive the SHAP values for the random forest (code below). Together with your results from Q1 and Q2, compare and contrast the random forest with the logistic model. Which model is more sensitive to which attributes?

b) **[Regular]** Show the SHAP waterfall plot of your patient for both models. Which explanation would you show to a doctor? Why? Assess permutation importance, PDP and SHAP on completeness, comprehensibility and stability.

In [ ]:
# SHAP for the random forest (exact and fast for trees)
explainer_rf = shap.TreeExplainer(rf)
sv_rf = explainer_rf(X_test)
if sv_rf.values.ndim == 3:       # one set of values per class: keep class 1 (heart disease)
    sv_rf = sv_rf[..., 1]

shap.plots.beeswarm(sv_rf, max_display=20)

**Your answers:**

a) 

b) 

## Q5) Find the hidden change

a) **[Advanced]** The data of your group has one hidden change compared to the original data set (`data/heart_original.csv`). For example, the effect of one attribute was made stronger, or one attribute was made to depend on another one. Use the methods of this lab on both data sets to find the change. Which attribute(s) were changed, and what kind of change is it? Show the plots that support your answer. Which method was most useful, and which one did not show the change?

In [ ]:
original = pd.read_csv("data/heart_original.csv")  # TODO: use the same arguments as for your own file
# Your analysis here

**Your answers:**

a) 